# Chapter 2: The Model Landscape

**From *AI Enterprise Architecture***

This notebook calls three different LLMs -- OpenAI GPT, Anthropic Claude, and Google Gemini --
with the same prompts and compares:

- **Response quality** (you be the judge)
- **Latency** (wall-clock time per call)
- **Token usage** (input and output)
- **Estimated cost** (based on current pricing)

**Key insight:** Model selection is a business decision with major cost, latency,
and quality implications. Enterprise architects must understand these trade-offs.

## Setup

Install the SDKs for all three providers.

In [ ]:
!pip install -q openai anthropic google-genai tabulate

## Configure API Keys

Set each provider's API key as an environment variable. In Google Colab you can
use the **Secrets** panel (key icon in the left sidebar) and then access them
with `userdata.get()`.

You need at least one key to run the notebook, but all three to see the full comparison.

In [ ]:
import os
import time
from tabulate import tabulate

# --- Uncomment these lines in Google Colab ---
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_KEY = os.environ.get("GOOGLE_API_KEY")

available = []
if OPENAI_KEY:
    available.append("OpenAI")
if ANTHROPIC_KEY:
    available.append("Anthropic")
if GOOGLE_KEY:
    available.append("Google")

if not available:
    raise ValueError(
        "No API keys found. Set at least one of: "
        "OPENAI_API_KEY, ANTHROPIC_API_KEY, GOOGLE_API_KEY"
    )

print(f"Available providers: {', '.join(available)}")

## Initialize Clients

We create a client for each available provider.

In [ ]:
openai_client = None
anthropic_client = None
google_client = None

if OPENAI_KEY:
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_KEY)

if ANTHROPIC_KEY:
    import anthropic
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

if GOOGLE_KEY:
    from google import genai
    google_client = genai.Client(api_key=GOOGLE_KEY)

print("Clients initialized.")

## Define Test Prompts

We use three prompts of increasing complexity to stress-test each model:

1. **Simple factual** -- a straightforward knowledge question
2. **Summarization** -- compress a paragraph into key points
3. **Complex reasoning** -- a multi-step logic problem

In [ ]:
PROMPTS = {
    "simple_factual": (
        "What is the CAP theorem in distributed systems? "
        "Explain in 2-3 sentences."
    ),
    "summarization": (
        "Summarize the following into 3 bullet points:\n\n"
        "Enterprise architecture (EA) is a discipline for proactively and holistically "
        "leading enterprise responses to disruptive forces by identifying and analyzing "
        "the execution of change toward desired business vision and outcomes. EA delivers "
        "value by presenting business and IT leaders with signature-ready recommendations "
        "for adjusting policies and projects to achieve targeted business outcomes that "
        "capitalize on relevant business disruptions. EA is used to steer decision-making "
        "toward the evolution of the future-state architecture."
    ),
    "complex_reasoning": (
        "A company has three microservices: Auth, Orders, and Inventory. "
        "Auth has 99.9% uptime. Orders depends on Auth and has 99.5% uptime "
        "(independent of Auth failures). Inventory depends on both Auth and Orders "
        "and has 99.0% uptime (independent of the others). "
        "What is the effective availability of the Inventory service from an "
        "end-user perspective? Show your reasoning step by step."
    ),
}

print(f"Defined {len(PROMPTS)} test prompts.")

## Define Caller Functions

Each function calls its respective API and returns a standardized result dict
containing the response text, latency, and token counts.

In [ ]:
def call_openai(prompt: str) -> dict:
    """Call OpenAI GPT-4o-mini and return standardized metrics."""
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = (time.time() - start) * 1000
    usage = response.usage
    return {
        "model": "gpt-4o-mini",
        "response": response.choices[0].message.content.strip(),
        "latency_ms": round(latency_ms),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
    }


def call_anthropic(prompt: str) -> dict:
    """Call Claude 3.5 Haiku and return standardized metrics."""
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-3-5-haiku-latest",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "model": "claude-3.5-haiku",
        "response": response.content[0].text.strip(),
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def call_google(prompt: str) -> dict:
    """Call Gemini 2.0 Flash and return standardized metrics."""
    start = time.time()
    response = google_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config={"max_output_tokens": 300},
    )
    latency_ms = (time.time() - start) * 1000
    usage = response.usage_metadata
    return {
        "model": "gemini-2.0-flash",
        "response": response.text.strip(),
        "latency_ms": round(latency_ms),
        "input_tokens": usage.prompt_token_count,
        "output_tokens": usage.candidates_token_count,
    }


print("Caller functions defined.")

## Run the Comparison

We call every available model with every prompt and collect the results.

In [ ]:
# Map provider names to their caller functions
callers = {}
if openai_client:
    callers["OpenAI"] = call_openai
if anthropic_client:
    callers["Anthropic"] = call_anthropic
if google_client:
    callers["Google"] = call_google

results = []  # list of dicts

for prompt_name, prompt_text in PROMPTS.items():
    for provider_name, call_fn in callers.items():
        print(f"  Calling {provider_name} with '{prompt_name}'...")
        try:
            result = call_fn(prompt_text)
            result["prompt_name"] = prompt_name
            result["provider"] = provider_name
            results.append(result)
        except Exception as e:
            print(f"    ERROR: {e}")
            results.append({
                "prompt_name": prompt_name,
                "provider": provider_name,
                "model": "N/A",
                "response": f"ERROR: {e}",
                "latency_ms": 0,
                "input_tokens": 0,
                "output_tokens": 0,
            })
        time.sleep(0.5)  # polite rate limiting

print(f"\nCollected {len(results)} results.")

## Cost Estimation

We estimate cost per request using approximate pricing (USD per 1M tokens).
These prices change -- always check the provider's pricing page for current rates.

In [ ]:
# Approximate pricing per 1M tokens (USD) -- update as needed
PRICING = {
    "gpt-4o-mini":       {"input": 0.15,  "output": 0.60},
    "claude-3.5-haiku":  {"input": 0.80,  "output": 4.00},
    "gemini-2.0-flash":  {"input": 0.10,  "output": 0.40},
}


def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Estimate cost in USD for a single API call."""
    if model not in PRICING:
        return 0.0
    p = PRICING[model]
    cost = (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000
    return cost


# Add cost to each result
for r in results:
    r["estimated_cost_usd"] = estimate_cost(
        r["model"], r["input_tokens"], r["output_tokens"]
    )

print("Cost estimates added.")

## Results: Comparison Table

This table shows every model-prompt combination with key metrics side by side.

In [ ]:
table_rows = []
for r in results:
    # Truncate response for the table
    preview = r["response"][:80].replace("\n", " ")
    if len(r["response"]) > 80:
        preview += "..."

    table_rows.append([
        r["model"],
        r["prompt_name"],
        preview,
        r["latency_ms"],
        r["input_tokens"],
        r["output_tokens"],
        f"${r['estimated_cost_usd']:.6f}",
    ])

headers = ["Model", "Prompt", "Response Preview", "Latency (ms)",
           "In Tokens", "Out Tokens", "Est. Cost"]

print(tabulate(table_rows, headers=headers, tablefmt="github"))

## Full Responses

Read the complete responses to judge quality for yourself.

In [ ]:
for r in results:
    print("=" * 70)
    print(f"Model: {r['model']}  |  Prompt: {r['prompt_name']}")
    print(f"Latency: {r['latency_ms']} ms  |  Tokens: {r['input_tokens']} in / {r['output_tokens']} out")
    print(f"Estimated cost: ${r['estimated_cost_usd']:.6f}")
    print("-" * 70)
    print(r["response"])
    print()

## Cost at Scale

What if you run 10,000 requests per day? Let's project monthly costs.

In [ ]:
DAILY_REQUESTS = 10_000
DAYS_PER_MONTH = 30

print(f"Projected monthly cost at {DAILY_REQUESTS:,} requests/day:\n")

# Average cost per model across all prompts
from collections import defaultdict

model_costs = defaultdict(list)
for r in results:
    model_costs[r["model"]].append(r["estimated_cost_usd"])

scale_rows = []
for model, costs in model_costs.items():
    avg_cost = sum(costs) / len(costs)
    monthly = avg_cost * DAILY_REQUESTS * DAYS_PER_MONTH
    scale_rows.append([model, f"${avg_cost:.6f}", f"${monthly:,.2f}"])

print(tabulate(
    scale_rows,
    headers=["Model", "Avg Cost/Request", "Monthly Cost (10k/day)"],
    tablefmt="github",
))

## Key Takeaways for Enterprise Architects

1. **No single "best" model.** The right choice depends on your requirements:
   quality, latency, cost, and data-residency constraints.

2. **Cost differences are dramatic at scale.** A 5x difference in per-request cost
   becomes hundreds of thousands of dollars per month at enterprise volumes.

3. **Latency varies significantly.** For real-time user-facing features, latency
   may matter more than quality.

4. **Build provider-agnostic abstractions.** Wrap model calls behind a common
   interface so you can swap providers without rewriting application code.

5. **Model selection is not one-and-done.** New models launch frequently. Build
   evaluation pipelines so you can continuously benchmark alternatives.